# Setup

### Import modules

In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm
from matplotlib.ticker import ScalarFormatter, StrMethodFormatter
from matplotlib import cm
from matplotlib.colors import Normalize
formatter = ScalarFormatter()
import re
import configparser
from glob import glob
import os
plt.rcParams.update({"text.usetex": True,})

import warnings
warnings.filterwarnings("ignore")

### Manual Modules

In [ ]:
from constants import *

# Functions

## Computations

In [ ]:
def OpenAp(ap_file, *args, **kwargs):
    #Open the ACS ap data, default separator is tab value
    return pd.read_csv(ap_file, *args, **kwargs)

def extract_absorption(
    ap,
    wl,
    start_wl=399,
    end_wl=701,
    first_column=None,
    last_column=None
): 

    ap = ap.iloc[:,first_column:last_column]
    
    wl_mask = (wl[0] > start_wl) & (wl[0] < end_wl)
    ap = ap.iloc[:,wl_mask.values]
    
    return ap, wl[wl_mask]

def pd2xr(pd_df, name='ap', dt_col='Time'): 
    
    #Wavelength = [floor(x) for x in pd_df.columns]
    Time = pd.to_datetime(pd_df.index.values , format='%Y-%m-%d %h:%M:%S')
    return xr.DataArray(data = pd_df.values, name = name, dims=['Time', 'Wavelength'],
                      coords={'Time':Time, 'Wavelength':pd_df.columns.values})

def StandardAp(ap): #Standardize each ap spectra 
    return (ap - ap.mean(dim='Wavelength'))/(ap.std(dim='Wavelength',ddof=1))

def read_EOF_U(EOF_U_format, PFT):
    
    EOF_U_fname_dict={}
    for pft_name in PFT:
        EOF_U_fname_dict[pft_name] = EOF_U_format.replace('*',pft_name)
    
    # Create EOF_U DataArray
    EOF_U = xr.Dataset()
    
    for key, value in EOF_U_fname_dict.items():
        
        data = pd.read_csv(value,index_col=0)
        
        EOF_U[key] = xr.DataArray(
            data = data, 
            dims = ['Wavelength','EOF'],
            coords={'EOF': [int(re.findall(r'\d+',i)[0]) for i in data.columns.values],
                    'Wavelength':data.index.values}
        )
        
        
    EOF_U = EOF_U.to_array(dim='PFT', name='EOF_U')
    
    return EOF_U


def NewEOFU(ap_r_standard, EOF_U): # Computing the New EOF U data
    return xr.dot(EOF_U, ap_r_standard, dims=["Wavelength", "Wavelength"])

def read_EOF_Coef(EOF_Coef_format, PFT):
    
    
    EOF_Coef_fname_dict={}
    for pft_name in PFT:
        EOF_Coef_fname_dict[pft_name] = EOF_Coef_format.replace('*',pft_name)

    # Create EOF_Coef DataArray
    EOF_Coef = []
    
    for key, value in EOF_Coef_fname_dict.items():
        
        data = pd.read_csv(value,index_col=0)
        
        data.rename(index={'(Intercept)': 'EOF0'}, inplace=True)
        data.index = data.index.str[3:].astype(int)
        
        EOF_U_da = xr.DataArray(
            data = data,
            dims = ['EOF','PFT'],
            coords= {'EOF': data.index.values,
                    'PFT': [key]})
        EOF_Coef.append(EOF_U_da)
        
    EOF_Coef = xr.concat(EOF_Coef, dim='PFT',join='outer')
    EOF_Coef = EOF_Coef.rename('EOF_Coef')
    return EOF_Coef

def PFTAll(new_EOF_U, EOF_coef_r): # Computing the PFT
    return (EOF_coef_r.fillna(0.).sel(EOF=0,drop=True) + 
            xr.dot(EOF_coef_r.fillna(0.).sel(EOF=slice(1,None)), new_EOF_U.fillna(0.), dims=['EOF','EOF']))

def PFTLimitations(PFT_concentrations_log,PFT_limits,pft): # Limiting the PFT Concentration ranges to the reliable values
    for i in range(len(pft)):
        PFT_concentrations_log[i] = PFT_concentrations_log.sel(PFT=pft[i]).where(PFT_concentrations_log.sel(PFT=pft[i])>np.log10(PFT_limits[pft[i]][0]),
                                                                         other=np.log10(PFT_limits[pft[i]][0])
                                                                 ).where(PFT_concentrations_log.sel(PFT=pft[i])<np.log10(PFT_limits[pft[i]][1]),
                                                                         other=np.log10(PFT_limits[pft[i]][1]))
    return PFT_concentrations_log


def binning(PFT_concentrations, freq='1min', valid_count=5):
    PFT_concentrations_mean = PFT_concentrations.resample(Time=freq).mean(dim='Time');
    PFT_concentrations_std = PFT_concentrations.resample(Time=freq).std(dim='Time',ddof=1);
    PFT_concentrations_count = PFT_concentrations.resample(Time=freq).count(dim='Time');

    print('Number of valid data points after binning:', np.count_nonzero((PFT_concentrations_count.sel(PFT=PFT[0])>valid_count).values))
    
    PFT_concentrations_mean_Valid = PFT_concentrations_mean.sel(Time=(PFT_concentrations_count.sel(PFT=PFT[0])>valid_count).values)
    PFT_concentrations_std_Valid = PFT_concentrations_std.sel(Time=(PFT_concentrations_count.sel(PFT=PFT[0])>valid_count).values)
    
    return PFT_concentrations_mean_Valid, PFT_concentrations_std_Valid

In [ ]:
def location(location_file, delimiter=",", datetimeformat='%m/%d/%Y %H:%M:%S'):
    dship = pd.read_csv(location_file ,sep=delimiter)
    dship['Time'] = pd.to_datetime(dship['Date']+ ' ' + dship['Time'], format=datetimeformat)
    dship = dship[['Time','Lat','Lon']]
    dship = dship.set_index('Time')
    dship_ds = xr.Dataset.from_dataframe(dship)
    return dship_ds

## Plot Functions

In [ ]:
def plot_wl_3d(ap_da, resample_rate, plot_dir):

    # resample the ap values to "resample_rate"
    ap_plot = ap_da.resample(Time=resample_rate).median()

    # Generate some sample data
    x = ap_plot.Wavelength.values
    z_values = ap_plot.Time.values
    y = ap_plot.values.T
    
    z_values = mdates.date2num(z_values)
    # Normalize z values to map them to the colormap
    norm = Normalize(vmin=np.nanmin(y), vmax=np.nanmax(y))  # Normalize based on y values
    cmap = cm.ocean_r  # You can change this to any colormap you like (e.g., 'plasma', 'inferno', etc.)

    
    # Create a figure and 3D axis
    fig, ax = plt.subplots(1, 1, figsize=(4, 4), subplot_kw={'projection': '3d'}, constrained_layout=True)

    # Plot each 2D line plot as a surface in the 3D space
    for i, z in enumerate(z_values):
        color = cmap(norm(np.nanmax(y[:, i])))  # Apply colormap based on z-value
        ax.plot(x, y[:,i], z, zdir='x', linewidth=0.8, color=color)

    # Set labels and title
    ax.set_box_aspect(None, zoom=0.9)
    # ax.set_xlabel('Time')
    ax.set_ylabel('Wavelength $[nm]$', labelpad=-2)
    ax.set_zlabel(r'$a_p [m^{-1}]$', labelpad=1)
    # ax.set_title(f'Absorption spectra ({resample_rate} resampled)')

    # Format the x-axis to display datetime values
    # ax.xaxis.set_major_locator(mdates.DayLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b %Y'))
    
    ax.tick_params(axis='x', pad=-7)
    ax.tick_params(axis='y', pad=-3)
    ax.tick_params(axis='z', pad=1)
    ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 0))
    ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 0))
    ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 0))

    # Add a legend
    ax.legend_ = None
    fig.autofmt_xdate()
    
    ax.view_init(elev=30, azim=320)

    # Show the plot
    # plt.show()

    plt.savefig(
        os.path.join(plot_dir,f'ACS_spectra_3D_{resample_rate}.png'),
        dpi=300,
        # bbox_inches='tight',
        transparent=True,
    )


In [ ]:
def plot_wl_spectra(ap_da, spectra_plot_start_time, spectra_plot_end_time, plot_dir):
    fig, ax = plt.subplots(1,1,figsize=(6,4))
    ap_da.sel(Time=slice(spectra_plot_start_time,spectra_plot_end_time,None)).plot.line(x='Wavelength', linewidth=1);
    
    # ap_da.plot.line(x='Wavelength');
    ax.legend_ = None
    ax.set_title(f'Absorption Spectra\n{spectra_plot_start_time} to {spectra_plot_end_time}')
    ax.set_ylabel(r'$a_p [m^{-1}]$')
    
    plt.savefig(os.path.join(plot_dir,f'ACS_spectra.png'),dpi=600,bbox_inches='tight')

In [ ]:
def plot_PFT_concentration(PFT_concentrations, PFT,plot_dir):
    pft_names = list(PFT_concentrations.data_vars)
    num_rows= len(pft_names)//2 + len(pft_names)%2
    fig, ax = plt.subplots(num_rows,2 , figsize=(15 , 1.5* len(pft_names)), constrained_layout=True, sharey=True)
    ax=ax.flatten()
    fig.supylabel(r'Chla concentrations [$mg/m^3$]', fontsize=14)
    for idx, pft_name in enumerate(pft_names):
        PFT_concentrations[pft_name].plot.scatter(ax=ax[idx], s=0.5, c='k',edgecolor='none')
        ax[idx].set_yscale('log')
        ax[idx].set_xlabel('')
        ax[idx].set_ylabel('')
        ax[idx].set_title(pft_name, fontsize=14)
        ax[idx].grid(alpha=0.3)
        ax[idx].set_yticks([0.001,0.01,0.1,1,10],[0.001,0.01,0.1,1,10])
    
    if len(pft_names)%2==1:
        fig.delaxes(ax[-1])
        
    plt.savefig(os.path.join(plot_dir,f'PFT_concentration.png'),dpi=600,bbox_inches='tight')

# Execute

### Configparser

In [ ]:
def main(
    ap_file,
    wl_file,
    first_column,
    last_column,
    start_wl,
    end_wl,
    EOF_U_format,
    EOF_Coef_format,
    do_binning,
    binning_freq,
    valid_number_binning,
    plot_dir,
    plot_spectra_3D,
    resample_rate,
    plot_spectra,
    spectra_plot_start_time,
    spectra_plot_end_time,
    plot_concentration,
    add_location,
    location_file,
    datetimeformat,
    delimiter,
    output_dir,
):
    
    ## open the absorption spectra
    ap = OpenAp(ap_file, sep=',',index_col=0,header=None)
    
    ## open wavelength data
    wl = OpenAp(wl_file, sep=',',header=None)

    ## Create header with wavelength
    ap_header = pd.concat((wl,wl),axis=0)[0]
    
    ap.columns = ap_header
    ap.index.name='datetime'
    ap.index = pd.to_datetime(ap.index)
    
    ## Only keep the absoprtion data and specific wavelength
    ap, wl = extract_absorption(
        ap,
        wl,
        start_wl=start_wl,
        end_wl=end_wl,
        first_column=first_column,
        last_column=last_column,
    )
    
    ## Convert from Pandas DataFrame to Xarray Dataset
    ap_da = pd2xr(ap)
    
    ## Standardize the dataset
    ap_da_standard = StandardAp(ap_da)
    
    ## read EOF_U files and create Xarray DataArray
    EOF_U = read_EOF_U(EOF_U_format, PFT=PFT)
    
    # Assigning the same wavelength name as ap to EOF_U for consistency
    EOF_U = EOF_U.assign_coords({'Wavelength':ap_da.Wavelength.values})
    
    # Computing the New EOF U
    new_EOF_U = NewEOFU(ap_da_standard, EOF_U)
    
    # read EOF_Coef files and create Xarray DataArray
    EOF_Coef = read_EOF_Coef(EOF_Coef_format, PFT)
    
    
    # Compure PFT Concentration
    PFT_concentrations_log = PFTAll(new_EOF_U, EOF_Coef)
    
    
    # Compute Prokaryotes Concentrations with summation of Cyano_noProchl and Prochl
    # Proka = np.log((np.exp(PFT_concentrations_log.sel(PFT='Cyano_noProchl'))+np.exp(PFT_concentrations_log.sel(PFT='Prochl')))).assign_coords({'PFT':'Proka'})
    # Attach Proka and remove Cyano_noProchl
    # PFT_concentrations_log = xr.concat([PFT_concentrations_log.drop_sel(PFT='Cyano_noProchl'),Proka],dim='PFT')
    
    
    ## Convert everything to log_10 instead of log_e
    PFT_concentrations_log = np.log10(np.exp(PFT_concentrations_log))
    
    if do_binning:
        PFT_concentrations_log ,PFT_concentrations_log_std = binning(PFT_concentrations_log, freq=binning_freq, valid_count=valid_number_binning)
    
    # Limiting the range of PFT concentrations
    PFT_concentrations_log = PFTLimitations(PFT_concentrations_log,PFT_limits,PFT_concentrations_log.PFT.values)
    
    # PFT_concentrations_log = PFT_concentrations_log.to_dataset(dim='PFT')
    # Compure the Absolute Concentrations
    PFT_concentrations = 10**PFT_concentrations_log
    
    PFT_concentrations_log.name='Chla concentrations Log'
    PFT_concentrations.name='Chla concentrations'
    
    PFT_concentrations_log.attrs['units'] = r'$\log_{10}[mg/m^3]$'
    PFT_concentrations.attrs['units'] = r'$mg/m^3$'
    
    PFT_concentrations = PFT_concentrations.to_dataset(dim='PFT')
    PFT_concentrations_log = PFT_concentrations_log.to_dataset(dim='PFT')
    PFT_concentrations_log_std = PFT_concentrations_log_std.to_dataset(dim='PFT')
    PFT_concentrations_log_std_rel = PFT_concentrations_log_std/abs(PFT_concentrations_log)*100
    
    # add location to the concentration file
    if add_location:
        ## Open the file
        dship_ds = location(location_file,delimiter=delimiter, datetimeformat=datetimeformat)

        ## choose the time that are shared with PFT
        dship_ds_c = dship_ds.sel(Time=PFT_concentrations_log.Time)

        PFT_concentrations_log['Lat'] = dship_ds_c['Lat']
        PFT_concentrations_log['Lon'] = dship_ds_c['Lon']
        PFT_concentrations_log = PFT_concentrations_log.set_coords(['Lat','Lon'])


        PFT_concentrations['Lat'] = dship_ds_c['Lat']
        PFT_concentrations['Lon'] = dship_ds_c['Lon']
        PFT_concentrations = PFT_concentrations.set_coords(['Lat','Lon'])
        
        
        PFT_concentrations_log_std['Lat'] = dship_ds_c['Lat']
        PFT_concentrations_log_std['Lon'] = dship_ds_c['Lon']
        PFT_concentrations_log_std = PFT_concentrations_log_std.set_coords(['Lat','Lon'])

        PFT_concentrations_log_std_rel['Lat'] = dship_ds_c['Lat']
        PFT_concentrations_log_std_rel['Lon'] = dship_ds_c['Lon']
        PFT_concentrations_log_std_rel = PFT_concentrations_log_std_rel.set_coords(['Lat','Lon'])

    if plot_spectra_3D:
        plot_wl_3d(ap_da,resample_rate, plot_dir)

    if plot_spectra:
        plot_wl_spectra(ap_da, spectra_plot_start_time, spectra_plot_end_time, plot_dir)

    if plot_concentration:
        plot_PFT_concentration(PFT_concentrations, PFT, plot_dir)

    PFT_concentrations.to_netcdf(os.path.join(output_dir,'PFT_concentration.nc'))
    PFT_concentrations_log.to_netcdf(os.path.join(output_dir,'PFT_concentration_log.nc'))
    PFT_concentrations_log_std.to_netcdf(os.path.join(output_dir,'PFT_concentrations_log_std.nc'))
    PFT_concentrations_log_std_rel.to_netcdf(os.path.join(output_dir,'PFT_concentrations_log_std_rel.nc'))
    
    return PFT_concentrations, PFT_concentrations_log,PFT_concentrations_log_std,PFT_concentrations_log_std_rel, EOF_Coef, new_EOF_U, EOF_U,ap_da_standard, ap_da, ap, wl

In [ ]:
config = configparser.ConfigParser()
config.read('./config_PS113-unc.ini')

# directory of ac-s raw data
ap_file     = config['acs_pft_retrieval']['ap_file']
wl_file     = config['acs_pft_retrieval']['wl_file']
first_column = config['acs_pft_retrieval'].getint('first_column')
last_column = config['acs_pft_retrieval'].getint('last_column')
start_wl = config['acs_pft_retrieval'].getfloat('start_wl')
end_wl = config['acs_pft_retrieval'].getfloat('end_wl')
EOF_U_format = config['acs_pft_retrieval']['EOF_U_format']
EOF_Coef_format = config['acs_pft_retrieval']['EOF_Coef_format']

#binning
do_binning=config['acs_pft_retrieval'].getboolean('do_binning')
binning_freq=config['acs_pft_retrieval']['binning_freq']
valid_number_binning=config['acs_pft_retrieval'].getint('valid_number_binning')

plot_dir = config['acs_pft_retrieval']['plot_dir']

plot_spectra_3D = config['acs_pft_retrieval'].getboolean('plot_spectra_3D')
resample_rate = config['acs_pft_retrieval']['resample_rate']

plot_spectra = config['acs_pft_retrieval'].getboolean('plot_spectra')
spectra_plot_start_time = np.datetime64(config['acs_pft_retrieval']['spectra_plot_start_time'])
spectra_plot_end_time = np.datetime64(config['acs_pft_retrieval']['spectra_plot_end_time'])
plot_concentration = config['acs_pft_retrieval'].getboolean('plot_concentration')

add_location=config['acs_pft_retrieval'].getboolean('add_location')
location_file=config['acs_pft_retrieval']['location_file']
datetimeformat=config['acs_pft_retrieval']['datetimeformat']
delimiter=config['acs_pft_retrieval']['delimiter']

output_dir = config['acs_pft_retrieval']['output_dir']
# o = config['acs_pft_retrieval'].getboolean('o')

In [ ]:
PFT_concentrations, PFT_concentrations_log, PFT_concentrations_log_std, PFT_concentrations_log_std_rel, EOF_Coef, new_EOF_U, EOF_U,ap_da_standard, ap_da, ap, wl = main(
    ap_file = ap_file,
    wl_file = wl_file,
    first_column=first_column,
    last_column=last_column,
    start_wl=start_wl,
    end_wl=end_wl,
    EOF_U_format=EOF_U_format,
    EOF_Coef_format=EOF_Coef_format,
    do_binning=do_binning,
    binning_freq=binning_freq,
    valid_number_binning=valid_number_binning,
    plot_dir=plot_dir,
    plot_spectra_3D=plot_spectra_3D,
    resample_rate=resample_rate,
    plot_spectra=plot_spectra,
    spectra_plot_start_time=spectra_plot_start_time,
    spectra_plot_end_time=spectra_plot_end_time,
    plot_concentration=plot_concentration,
    add_location=add_location,
    location_file=location_file,
    datetimeformat=datetimeformat,
    delimiter=delimiter,
    output_dir=output_dir,
)